In [1]:
import os
import pandas as pd
import numpy as np

In [11]:
df = pd.read_pickle("SCOP_taxonomy.pkl")
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,cellular organisms Bacteria Terrabacteria gr...
11,a.1.1.1,5885,cellular organisms Eukaryota Sar Alveolata ...
14,a.1.1.1,3054,cellular organisms Eukaryota Viridiplantae ...
17,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
34,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
...,...,...,...
362200,k.13.1.1,1280,cellular organisms Bacteria Terrabacteria gr...
362462,k.37.1.1,32630,other entries other sequences artificial seq...
362467,k.37.1.1,32644,unclassified entries unclassified sequences
362475,k.37.1.1,32630,other entries other sequences artificial seq...


In [12]:
#keep proteins in the first 7 SCOPe classes
df = df[df['fold'].str[0].isin(['a','b','c','d','e','f','g'])]
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,cellular organisms Bacteria Terrabacteria gr...
11,a.1.1.1,5885,cellular organisms Eukaryota Sar Alveolata ...
14,a.1.1.1,3054,cellular organisms Eukaryota Viridiplantae ...
17,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
34,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
...,...,...,...
353272,g.101.1.1,4932,cellular organisms Eukaryota Opisthokonta F...
353275,g.101.1.1,9606,cellular organisms Eukaryota Opisthokonta M...
353285,g.102.1.1,9606,cellular organisms Eukaryota Opisthokonta M...
353302,g.102.1.0,9606,cellular organisms Eukaryota Opisthokonta M...


In [13]:
#check the taxonomy for the string 'therm'
thermophile_taxids = df[df['taxonomy'].str.contains('therm',case=False)]['taxid'].unique()
df.loc[:,'therm'] = False
df.loc[df['taxid'].isin(thermophile_taxids), 'therm'] = True
df

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_15684\2376305915.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'therm'] = False


,fold,taxid,taxonomy,therm
9,a.1.1.1,1423,cellular organisms Bacteria Terrabacteria gr...,False
11,a.1.1.1,5885,cellular organisms Eukaryota Sar Alveolata ...,False
14,a.1.1.1,3054,cellular organisms Eukaryota Viridiplantae ...,False
17,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...,False
34,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...,False
...,...,...,...,...
353272,g.101.1.1,4932,cellular organisms Eukaryota Opisthokonta F...,False
353275,g.101.1.1,9606,cellular organisms Eukaryota Opisthokonta M...,False
353285,g.102.1.1,9606,cellular organisms Eukaryota Opisthokonta M...,False
353302,g.102.1.0,9606,cellular organisms Eukaryota Opisthokonta M...,False


### Keep all the prokaryotic sources

In [14]:
df.loc[df['taxonomy'].str.contains("Bacteria"),  "taxonomy"] = "Bacteria"
df.loc[df['taxonomy'].str.contains("Archaea"),   "taxonomy"] = "Archaea"
df.loc[df['taxonomy'].str.contains("Eukaryota"), "taxonomy"] = "Eukaryota"

In [19]:
df = df[(df['taxonomy']=="Bacteria") | (df['taxonomy']=="Archaea")]
df

,fold,taxid,taxonomy,therm
9,a.1.1.1,1423,Bacteria,False
17,a.1.1.1,1773,Bacteria,False
34,a.1.1.1,1773,Bacteria,False
59,a.1.1.1,1148,Bacteria,False
68,a.1.1.1,1392,Bacteria,False
...,...,...,...,...
352764,g.74.1.1,562,Bacteria,False
352768,g.74.1.1,727,Bacteria,False
352868,g.81.1.1,1423,Bacteria,False
352871,g.81.1.1,562,Bacteria,False


### Keep all families that have an entry from a thermophilic source + entry from non-thermophilic source

In [20]:
keep_families = []
for family in df['fold'].unique():
    if not df[df['fold']==family]['therm'].all():
        keep_families.append(family)

In [22]:
df = df[df['fold'].isin(keep_families)]
df

,fold,taxid,taxonomy,therm
9,a.1.1.1,1423,Bacteria,False
17,a.1.1.1,1773,Bacteria,False
34,a.1.1.1,1773,Bacteria,False
59,a.1.1.1,1148,Bacteria,False
68,a.1.1.1,1392,Bacteria,False
...,...,...,...,...
352764,g.74.1.1,562,Bacteria,False
352768,g.74.1.1,727,Bacteria,False
352868,g.81.1.1,1423,Bacteria,False
352871,g.81.1.1,562,Bacteria,False


### The index for the dataframe corresponds to the line numbers in the dir.des.scope file, so can use the line numbers to get the extra info

In [24]:
all_sp_lines = df.index.astype('int')
fold = []
dom  = []
sp   = []
boundary = []
with open("./data/dir.des.scope.2.08-stable.txt","r") as f:
    i = 0
    for line_no, line in enumerate(f):
        if line_no==all_sp_lines[i]:
            species = df.loc[str(line_no)]['taxid']
            i+=1
        line = line.split()
        if line[1] == "px":

            fold.append(line[2])
            dom.append(line[3])
            boundary.append("".join(line[4:]))
            sp.append(species)            
        if i>=len(all_sp_lines):
            break

all_domain_sp = pd.DataFrame ({"fold": fold, "domain": dom, "taxid":sp, "boundary":boundary})
all_domain_sp['PDB'] = all_domain_sp['domain'].str[1:5].str.upper() + "_" + all_domain_sp['domain'].str[5].str.upper()
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,1423,1dlwA:,1DLW_A
2,a.1.1.1,d1uvya_,1423,1uvyA:,1UVY_A
3,a.1.1.1,d1dlya_,1423,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,1423,1uvxA:,1UVX_A
...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1wfhA:8-58,1WFH_A
302294,g.80.1.1,d1wfla1,727,1wflA:8-68,1WFL_A
302295,g.81.1.1,d1vzya2,1423,1vzyA:234-290,1VZY_A
302296,g.81.1.1,d1vzyb2,1423,1vzyB:234-286,1VZY_B


### retain the PDB ID's present in the pdb70 dataset

In [31]:
with open("./data/pdb70_a3m.FFINDEX", "r") as f:
     pdb70 = [x.split()[0] for x in f.readlines()]

In [33]:
all_domain_sp = all_domain_sp[all_domain_sp['PDB'].isin(pdb70)]
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,1423,1dlwA:,1DLW_A
3,a.1.1.1,d1dlya_,1423,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,1423,1uvxA:,1UVX_A
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B
...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1wfhA:8-58,1WFH_A
302294,g.80.1.1,d1wfla1,727,1wflA:8-68,1WFL_A
302295,g.81.1.1,d1vzya2,1423,1vzyA:234-290,1VZY_A
302296,g.81.1.1,d1vzyb2,1423,1vzyB:234-286,1VZY_B


### Export this list of PDB ID's to then filter out any sequences with similarity to training data, and pairwise sequence identity from PISCES web server

In [34]:
with open("./data/thermophile_export_list.txt","w") as f:
    for x in all_domain_sp['PDB']:
        f.write(x + "\n")

### After filtering the list of PDB chains, import filtered list and use subset of dataframe

In [39]:
with open("./data/thermophile_import_list.txt", "r") as f:
    keep_chains = f.readlines()

all_domain_sp = all_domain_sp[all_domain_sp['PDB'].isin(keep_chains)]
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,1423,1dlwA:,1DLW_A
3,a.1.1.1,d1dlya_,1423,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,1423,1uvxA:,1UVX_A
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B
...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1wfhA:8-58,1WFH_A
302294,g.80.1.1,d1wfla1,727,1wflA:8-68,1WFL_A
302295,g.81.1.1,d1vzya2,1423,1vzyA:234-290,1VZY_A
302296,g.81.1.1,d1vzyb2,1423,1vzyB:234-286,1VZY_B


In [43]:
taxdomain   = dict(zip(df.taxid,df.taxonomy))
thermophile = dict(zip(df.taxid,df.therm)) 

In [50]:
all_domain_sp['therm']    = [thermophile[x] for x in all_domain_sp['taxid']]
all_domain_sp['tax_domain'] = [taxdomain[x] for x in all_domain_sp['taxid']]
all_domain_sp

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_15684\2239180195.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_domain_sp['therm']    = [thermophile[x] for x in all_domain_sp['taxid']]
C:\Users\Mohan2\AppData\Local\Temp\ipykernel_15684\2239180195.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_domain_sp['tax_domain'] = [taxdomain[x] for x in all_domain_sp['taxid']]


,fold,domain,taxid,boundary,PDB,therm,taxonomy,tax_domain
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A,False,Bacteria,Bacteria
1,a.1.1.1,d1dlwa_,1423,1dlwA:,1DLW_A,False,Bacteria,Bacteria
3,a.1.1.1,d1dlya_,1423,1dlyA:,1DLY_A,False,Bacteria,Bacteria
4,a.1.1.1,d1uvxa_,1423,1uvxA:,1UVX_A,False,Bacteria,Bacteria
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B,False,Bacteria,Bacteria
...,...,...,...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1wfhA:8-58,1WFH_A,False,Bacteria,Bacteria
302294,g.80.1.1,d1wfla1,727,1wflA:8-68,1WFL_A,False,Bacteria,Bacteria
302295,g.81.1.1,d1vzya2,1423,1vzyA:234-290,1VZY_A,False,Bacteria,Bacteria
302296,g.81.1.1,d1vzyb2,1423,1vzyB:234-286,1VZY_B,False,Bacteria,Bacteria


### double check that all families still have entries from thermophile and non-thermophiles

In [46]:
keep_families = []
for family in all_domain_sp['fold'].unique():
    if not all_domain_sp[all_domain_sp['fold']==family]['therm'].all():
        keep_families.append(family)

all_domain_sp = all_domain_sp[all_domain_sp['fold'].isin(keep_families)]
all_domain_sp

,fold,domain,taxid,boundary,PDB,therm,taxonomy
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A,False,Bacteria
1,a.1.1.1,d1dlwa_,1423,1dlwA:,1DLW_A,False,Bacteria
3,a.1.1.1,d1dlya_,1423,1dlyA:,1DLY_A,False,Bacteria
4,a.1.1.1,d1uvxa_,1423,1uvxA:,1UVX_A,False,Bacteria
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B,False,Bacteria
...,...,...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1wfhA:8-58,1WFH_A,False,Bacteria
302294,g.80.1.1,d1wfla1,727,1wflA:8-68,1WFL_A,False,Bacteria
302295,g.81.1.1,d1vzya2,1423,1vzyA:234-290,1VZY_A,False,Bacteria
302296,g.81.1.1,d1vzyb2,1423,1vzyB:234-286,1VZY_B,False,Bacteria


In [51]:
all_domain_sp.to_pickle("thermophile2.pkl")

In [48]:
len(all_domain_sp[all_domain_sp['therm']]['taxid'].unique())

176

In [49]:
len(all_domain_sp[~all_domain_sp['therm']]['taxid'].unique())

1465